<a href="https://colab.research.google.com/github/Shanmukh-dev/Custom-GPT/blob/main/CustomGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
from torch import nn
import math
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Transformer

In [7]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads):
    super().__init__()

    assert d_model % n_heads == 0

    self.d_model = d_model
    self.n_heads = n_heads
    self.head_dim = d_model // n_heads

    self.query = nn.Linear(in_features=d_model, out_features=d_model).to(device)
    self.key = nn.Linear(in_features=d_model, out_features=d_model).to(device)
    self.value = nn.Linear(in_features=d_model, out_features=d_model).to(device)

    self.out_proj = nn.Linear(d_model, d_model).to(device)

  def forward(self, X):
    X = X.to(device)
    B, T, C = X.shape
    Q, K, V = self.query(X), self.key(X), self.value(X)

    Q = Q.view(B, T, self.n_heads, self.head_dim).to(device)
    K = K.view(B, T, self.n_heads, self.head_dim).to(device)
    V = V.view(B, T, self.n_heads, self.head_dim).to(device)

    Q = Q.transpose(1, 2)
    K = K.transpose(1, 2)
    V = V.transpose(1, 2)

    scores = Q @ K.transpose(-1, -2)
    scores = scores / math.sqrt(self.head_dim)


    causal_mask = torch.tril(torch.ones(T, T)).to(device)
    scores = scores.masked_fill(causal_mask == 0, float("-inf")).to(device)

    attn_weights = torch.softmax(scores, dim = -1).to(device)

    attn = attn_weights @ V


    attn = attn.transpose(1, 2)

    attn = attn.contiguous().view(B, T, self.d_model).to(device)

    attn = self.out_proj(attn)
    return attn




class MLP(nn.Module):
  def __init__(self, input_dimensions, hidden_layers, output_dimensions) -> None:
    super().__init__()

    self.mpl_layer = nn.Sequential(
        nn.Linear(in_features=input_dimensions, out_features=hidden_layers).to(device),
        nn.GELU().to(device),
        nn.Linear(in_features=hidden_layers, out_features=output_dimensions).to(device)
    ).to(device)


  def forward(self, X):
    X = X.to(device)
    return self.mpl_layer(X)


class TransformerBlock(nn.Module):
  def __init__(self, d_model, n_heads, hidden_layers):
    super().__init__()

    self.multi_head_attention = MultiHeadAttention(d_model, n_heads).to(device)
    self.mlp = MLP(d_model, hidden_layers, d_model).to(device)

    self.ln1 = nn.LayerNorm(d_model).to(device)
    self.ln2 = nn.LayerNorm(d_model).to(device)


  def forward(self, X):
    X = X.to(device)

    normalized_X = self.ln1(X)
    attention_weights = self.multi_head_attention(normalized_X)
    attention_weights = X + attention_weights


    normalized_attention_weights = self.ln2(attention_weights).to(device)
    mlp_output = self.mlp(normalized_attention_weights)

    output = attention_weights + mlp_output

    return output




# GPT Modle


In [8]:
class CustomGPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.token_embeddings = nn.Embedding(config.vocab_size, config.d_model)

    self.positional_embeddings = nn.Embedding(config.block_size, config.d_model)


    self.transformer_blocks = nn.ModuleList(
        [
            TransformerBlock(config.d_model, config.n_heads, config.hidden_layers)
            for _ in range(config.n_layers)

        ]
    )


    self.layer_norm = nn.LayerNorm(config.d_model)
    self.lm_head = nn.Linear(config.d_model, config.vocab_size)
    self.config = config


  def forward(self, idx):
    B, T = idx.shape

    tok_embd = self.token_embeddings(idx)
    pos = torch.arange(T, device=device)

    pos_embd = self.positional_embeddings(pos)

    X = tok_embd + pos_embd

    for block in self.transformer_blocks:
      X = block(X)

    X = self.layer_norm(X)

    logits = self.lm_head(X)

    return logits


  @torch.no_grad()
  def generate(self, idx, max_new_tokens=500):
    for _ in range(max_new_tokens):
      idx_cont = idx[:, -self.config.block_size:]

      logits = self(idx_cont)

      logits = logits[:, -1, :]

      probs = torch.softmax(logits, dim=-1)
      new_tokens = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, new_tokens), dim=1)

    return idx


# The dataset and confg

In [9]:
from datasets import load_dataset

ds = load_dataset("PleIAs/SYNTH", split="train", streaming = True)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/500 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/500 [00:00<?, ?it/s]

In [10]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

tokens = []
target_tokens = 10_000_000

for sample in ds:
    text = (
        sample["query"]+
        sample["query_seed_text"]+
        sample["synthetic_reasoning"]+
        sample["synthetic_answer"]
    )

    ids = tokenizer.encode(text)

    tokens.extend(ids)


    if len(tokens) >= target_tokens:
        break


print(len(tokens))

10001401


In [11]:
from torch.utils.data import Dataset, DataLoader

data = torch.tensor(tokens, dtype=torch.long, device=device)
print(len(data))


train_split = int(0.9*len(data))
train_data = data[:train_split]
test_data = data[train_split:]
print("Train data length:", len(train_data))
print("Test data length:", len(test_data))

10001401
Train data length: 9001260
Test data length: 1000141


In [12]:
class GPTConfig:
  vocab_size = tokenizer.n_vocab
  block_size = 256

  d_model = 256
  hidden_layers = 1024
  n_heads = 4
  n_layers = 6

In [13]:
class SynthDataset(Dataset):
  def __init__(self, data, block_size):
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self, idx):
    x = data[idx:idx+self.block_size]
    y = data[idx+1:idx+self.block_size+1]
    return x, y

In [14]:
train_dataset = SynthDataset(train_data, GPTConfig.block_size)
test_dataset = SynthDataset(test_data, GPTConfig.block_size)

In [15]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [16]:
tokenizer.n_vocab

50257

# The training loop

In [24]:
from torch.amp import GradScaler, autocast

model = CustomGPT(GPTConfig)

if torch.cuda.device_count() > 1:
  print(f"Using {torch.cuda.device_count()} GPUs.")
  model = nn.DataParallel(model)

model.to(device)

scaler = GradScaler()

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4)

Using 2 GPUs.


In [25]:
from tqdm.auto import tqdm
steps = 50000

torch.manual_seed(42)
train_iter = iter(train_dataloader)
for step in tqdm(range(1, steps+1)):
  try:
    X_train, y_train = next(train_iter)
  except StopIteration:
    train_iter = iter(train_dataloader)
    X_train, y_train = next(train_iter)

  X_train, y_train = X_train.to(device), y_train.to(device)

  model.train()
  optimizer.zero_grad()
  with autocast(device):

    logits = model(X_train)

    train_loss = loss_fn(logits.view(-1, logits.size(-1)), y_train.view(-1))

  scaler.scale(train_loss).backward()
  scaler.step(optimizer)
  scaler.update()
  # optimizer.step()

  if step % 10000 == 0:
    model.eval()
    test_loss = 0
    with torch.inference_mode():

      for idx, (X_test, y_test) in tqdm(enumerate(test_dataloader)):
        if idx == 10:
          break
        X_test, y_test = X_test.to(device), y_test.to(device)
        test_logits = model(X_test)

        curr_loss = loss_fn(test_logits.view(-1, test_logits.size(-1)), y_test.view(-1))

        test_loss += curr_loss.item()


    test_loss = test_loss / 10

    print(f"Step: {step} | Training loss: {train_loss} | Testing loss: {test_loss}")





  0%|          | 0/50000 [00:00<?, ?it/s]

0it [00:00, ?it/s]

Step: 10000 | Training loss: 3.787724018096924 | Testing loss: 3.8522677183151246


0it [00:00, ?it/s]

Step: 20000 | Training loss: 3.2390811443328857 | Testing loss: 3.253386878967285


...

Step: 30000 | Training loss: 3.0798306465148926 | Testing loss: 3.050572371482849


0it [00:00, ?it/s]

Step: 40000 | Training loss: 2.968963623046875 | Testing loss: 2.8173280239105223


0it [00:00, ?it/s]

Step: 50000 | Training loss: 2.6730172634124756 | Testing loss: 2.7031943798065186


In [30]:
torch.save(model.state_dict(), "custom_gpt.pth")
print(model.state_dict())

OrderedDict({'module.token_embeddings.weight': tensor([[ 0.5173, -1.0625, -0.0809,  ...,  1.2041, -0.7221, -0.1296],
        [-1.0898,  0.4598, -0.0439,  ...,  0.7971,  0.3272,  0.3385],
        [ 1.5745, -0.1575, -0.6893,  ..., -0.3505, -0.7100, -0.4380],
        ...,
        [-0.2202, -0.5221, -0.0797,  ..., -0.4713, -0.5579, -0.2061],
        [-0.3566,  0.6046, -0.1711,  ...,  0.6990, -1.0015, -1.5099],
        [ 0.2710,  0.7927,  0.0631,  ...,  1.5534, -0.4368,  0.9052]],
       device='cuda:0'), 'module.positional_embeddings.weight': tensor([[ 1.0850, -1.3798, -0.2017,  ..., -0.4876,  0.0975,  1.7922],
        [ 0.8058, -0.2323, -0.3468,  ..., -0.0606,  1.7217,  2.0302],
        [-0.0193, -0.6114, -0.9169,  ...,  1.4927,  0.0216,  0.4758],
        ...,
        [ 1.0497, -0.1763,  0.3640,  ...,  0.8976, -0.6184,  0.8118],
        [ 0.0971,  0.7973,  0.4138,  ...,  0.4305,  0.3710, -1.6910],
        [ 0.4190, -0.2344, -0.6805,  ..., -0.5881,  0.0874,  1.3378]],
       device='cuda:0

In [32]:
test_model = CustomGPT(GPTConfig)

state_dict = torch.load("custom_gpt.pth", map_location=device)
state_dict = {key.removeprefix("module."): value for key, value in state_dict.items()}
test_model.load_state_dict(state_dict)
test_model.to(device)

test_model.eval()



CustomGPT(
  (token_embeddings): Embedding(50257, 256)
  (positional_embeddings): Embedding(256, 256)
  (transformer_blocks): ModuleList(
    (0-5): 6 x TransformerBlock(
      (multi_head_attention): MultiHeadAttention(
        (query): Linear(in_features=256, out_features=256, bias=True)
        (key): Linear(in_features=256, out_features=256, bias=True)
        (value): Linear(in_features=256, out_features=256, bias=True)
        (out_proj): Linear(in_features=256, out_features=256, bias=True)
      )
      (mlp): MLP(
        (mpl_layer): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1024, out_features=256, bias=True)
        )
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_featur

In [47]:
context = torch.tensor(tokenizer.encode("""Here's the explanation for the Newton's Three Laws of motion."""), dtype=torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    generated = test_model.generate(context, max_new_tokens=256)

In [48]:
print(tokenizer.decode(generated[0].tolist()))

Here's the explanation for the Newton's Three Laws of motion. It is equivalent that of Carlo Gott Puynds, a given form that Gauge, is no specific that can be seen independently of a following well-characteristic phenomenon.

Think of it that restraint is a state of action in the language of an ecosystem, and critic in the logic of the pursuit of the relationship between them is, a territory. Chemists disagree about this and emergent property: on the other hand, went on to step forward in revelation; a guided agency and the Court of Wendy Smith decided to take the report. A difference between the internal thinking of forcing of the more transient motion, was much more complex and difficult than the reverseine principle apparent.

When you follow a link between "Friedman said" could be famous for self-determination and that Becker Joseph claimed he had refused to use his general theory as "human touch" or "person" the sense of a proposition describing the success seeks to suppressau. In 

In [50]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 30,586,449
Trainable Parameters: 30,586,449
